## 02 — Data Preprocessing


In [19]:
import os
import sys
import pyspark
from pyspark.ml.feature import StringIndexer
from pyspark.sql.functions import lit, trim
from sklearn.model_selection import train_test_split
import pandas as pd


In [20]:
# Fix Windows environment: force correct SPARK_HOME and set HADOOP_HOME
# Must be done BEFORE SparkSession is created (JVM reads these at startup)
os.environ["SPARK_HOME"]  = os.path.dirname(pyspark.__file__)
os.environ["HADOOP_HOME"] = r"C:\hadoop"

print("SPARK_HOME :", os.environ["SPARK_HOME"])
print("HADOOP_HOME:", os.environ["HADOOP_HOME"])

sys.path.append(os.path.abspath(".."))

from pyspark.sql import SparkSession
from pyspark.sql.functions import col, lit, isnan, when, count as spark_count
from pyspark.ml.feature import VectorAssembler, MinMaxScaler

SPARK_HOME : c:\Users\Mariam\anaconda3\envs\spotify-recommender\Lib\site-packages\pyspark
HADOOP_HOME: C:\hadoop


In [21]:
spark = SparkSession.builder \
    .appName("SpotifyPreprocessing") \
    .master("local[*]") \
    .config("spark.driver.memory", "4g") \
    .getOrCreate()

spark.sparkContext.setLogLevel("ERROR")
print("Spark version:", spark.version)

Spark version: 3.5.8


2.1 - Load Data


In [22]:


DATA_DIR = "../data/raw"

datasets = {
    "spotify_tracks":    f"{DATA_DIR}/spotify_tracks/dataset.csv",
    "genres_v2":         f"{DATA_DIR}/dataset_of_songs/genres_v2.csv",
    "audio_apr2019":     f"{DATA_DIR}/spotify_audio_features/SpotifyAudioFeaturesApril2019.csv",
    "audio_nov2018":     f"{DATA_DIR}/spotify_audio_features/SpotifyAudioFeaturesNov2018.csv",
    "ultimate_tracks":   f"{DATA_DIR}/ultimate_spotify_tracks/SpotifyFeatures.csv",
}

dfs = {}
for name, path in datasets.items():
    dfs[name] = spark.read.csv(path, header=True, inferSchema=True)
    print(f"{name}: {dfs[name].count()} rows × {len(dfs[name].columns)} cols")
    

spotify_tracks: 114000 rows × 21 cols
genres_v2: 42305 rows × 22 cols
audio_apr2019: 130663 rows × 17 cols
audio_nov2018: 116372 rows × 17 cols
ultimate_tracks: 232725 rows × 18 cols


2.2 - Standardize Schemas


In [23]:


AUDIO_FEATURES = [
    "danceability", "energy", "loudness", "speechiness",
    "acousticness", "instrumentalness", "liveness", "valence", "tempo"
]

def select_standard(df, track_id, track_name, artist_name, genre, popularity):
    return df.select(
        col(track_id).alias("track_id"),
        col(track_name).alias("track_name"),
        col(artist_name).alias("artist_name"),
        col(genre).alias("genre"),
        col(popularity).cast("double").alias("popularity"),
        *[col(f).cast("double") for f in AUDIO_FEATURES]
    )

# df1 — spotify_tracks
df1 = select_standard(dfs["spotify_tracks"],
    "track_id", "track_name", "artists", "track_genre", "popularity")

# df2 — genres_v2
df2 = dfs["genres_v2"].select(
    col("id").alias("track_id"),
    col("song_name").alias("track_name"),
    lit(None).cast("string").alias("artist_name"),
    col("genre").alias("genre"),
    lit(None).cast("double").alias("popularity"),
    *[col(f).cast("double") for f in AUDIO_FEATURES]
)

# df3 — audio_apr2019
df3 = dfs["audio_apr2019"].select(
    col("track_id"), col("track_name"), col("artist_name"),
    lit(None).cast("string").alias("genre"),
    col("popularity").cast("double"),
    *[col(f).cast("double") for f in AUDIO_FEATURES]
)

# df4 — audio_nov2018
df4 = dfs["audio_nov2018"].select(
    col("track_id"), col("track_name"), col("artist_name"),
    lit(None).cast("string").alias("genre"),
    col("popularity").cast("double"),
    *[col(f).cast("double") for f in AUDIO_FEATURES]
)

# df5 — ultimate_tracks
df5 = dfs["ultimate_tracks"].select(
    col("track_id"), col("track_name"), col("artist_name"),
    col("genre"), col("popularity").cast("double"),
    *[col(f).cast("double") for f in AUDIO_FEATURES]
)

# df6 — spotify_2023
df_2023_raw = spark.read.csv(
    f"{DATA_DIR}/spotify_2023/spotify_data_12_20_2023.csv",
    header=True, inferSchema=True
)
df6 = df_2023_raw.select(
    col("track_id"), col("track_name"),
    col("artist_0").alias("artist_name"),
    col("genre_0").alias("genre"),
    col("track_popularity").cast("double").alias("popularity"),
    *[col(f).cast("double") for f in AUDIO_FEATURES]
)

# Note: MSD (df_msd_spark) is loaded above for course requirements
# but kept separate — not merged here due to missing Spotify features

print("Individual dataset sizes:")
for name, d in [("spotify_tracks", df1), ("genres_v2", df2), ("audio_apr2019", df3),
                ("audio_nov2018", df4), ("ultimate_tracks", df5), ("spotify_2023", df6)]:
    print(f"  {name}: {d.count():,} rows")

Individual dataset sizes:
  spotify_tracks: 114,000 rows
  genres_v2: 42,305 rows
  audio_apr2019: 130,663 rows
  audio_nov2018: 116,372 rows
  ultimate_tracks: 232,725 rows
  spotify_2023: 375,141 rows


In [24]:
# Union 6 Spotify datasets (MSD kept separate)
songs_df_merged = (df1.unionByName(df2).unionByName(df3)
               .unionByName(df4).unionByName(df5).unionByName(df6))
print(f"Total rows after union: {songs_df_merged.count():,}")

Total rows after union: 1,011,206


In [25]:
# Drop rows missing any audio feature (metadata nulls like genre/artist are acceptable)
songs_df_clean = songs_df_merged.dropna(subset=AUDIO_FEATURES)

# Deduplicate: same track_id = same song, keep first occurrence
songs_df_clean = songs_df_clean.dropDuplicates(["track_id"])

# Secondary dedup: same track_name + artist (catches tracks with different IDs across datasets)
songs_df_clean = songs_df_clean.dropDuplicates(["track_name", "artist_name"])

print(f"After dropping null audio features: {songs_df_clean.count()}")
print(f"Removed {songs_df_merged.count() - songs_df_clean.count()} duplicate/null rows")

After dropping null audio features: 697392
Removed 313814 duplicate/null rows


2.3 - Feature Selection


In [26]:
METADATA_COLS = ["track_id", "track_name", "artist_name", "genre", "popularity"]

songs_df_selected = songs_df_clean.select(METADATA_COLS + AUDIO_FEATURES)
songs_df_selected.show(5)
print(f"Final shape: {songs_df_selected.count()} rows × {len(songs_df_selected.columns)} cols")

+--------------------+--------------------+-----------------+-----+----------+------------+------+-------------------+-----------+------------+----------------+--------+------------------+-------+
|            track_id|          track_name|      artist_name|genre|popularity|danceability|energy|           loudness|speechiness|acousticness|instrumentalness|liveness|           valence|  tempo|
+--------------------+--------------------+-----------------+-----+----------+------------+------+-------------------+-----------+------------+----------------+--------+------------------+-------+
|0wtpkz93wATDkUExJ...|""" La Traviata "...|     Maria Callas|Opera|      31.0|       0.364| 0.275|            -11.832|      0.043|       0.993|          0.0284|   0.293|            0.0394| 86.096|
|6YQUuoMnRIMaOmouY...|            """99"""|   Barns Courtney| NULL|      69.0|       0.552| 0.804|-4.2989999999999995|     0.0303|     0.00598|             0.0|   0.111|0.7140000000000001|  95.98|
|4oDWA9n9KbHyrM

In [27]:
# Assemble audio features into a single vector column for ML algorithms
assembler = VectorAssembler(inputCols=AUDIO_FEATURES, outputCol="raw_features")
songs_df_assembled = assembler.transform(songs_df_selected)


# Normalize audio features to [0, 1] range using MinMaxScaler
scaler = MinMaxScaler(inputCol="raw_features", outputCol="scaled_features")
scaler_model = scaler.fit(songs_df_assembled)


# Apply the scaler to the data and drop the intermediate raw_features column
songs_df_normalized = scaler_model.transform(songs_df_assembled).drop("raw_features")
songs_df_normalized.select(METADATA_COLS + ["scaled_features"]).show(5, truncate=False)



+----------------------+-------------------------------------------------------------------+-----------------+-----+----------+-----------------------------------------------------------------------------------------------------------------------------------------+
|track_id              |track_name                                                         |artist_name      |genre|popularity|scaled_features                                                                                                                          |
+----------------------+-------------------------------------------------------------------+-----------------+-----+----------+-----------------------------------------------------------------------------------------------------------------------------------------+
|0wtpkz93wATDkUExJVuXEl|""" La Traviata "" : Amami Alfredo (Act II) - Digitally Remastered"|Maria Callas     |Opera|31.0      |[0.017333333333333333,1.5708361417979505E-6,0.7464203805863757,0.0086,0.999

2.4 - Save new dataset


In [28]:


PROCESSED_DIR = "../data/processed"
SPLIT_DIR     = "../outputs/train_test_split"

os.makedirs(PROCESSED_DIR, exist_ok=True)
os.makedirs(SPLIT_DIR,     exist_ok=True)

# Save via pandas — bypasses Hadoop/winutils entirely on Windows
# Save df_selected (clean data, individual columns) — normalized vector can be rebuilt in notebook 04
songs_preprocessed_df = songs_df_selected.toPandas()
songs_preprocessed_df.to_csv(f"{PROCESSED_DIR}/songs_cleaned.csv", index=False)

print(f"Saved {len(songs_preprocessed_df):,} rows → {PROCESSED_DIR}/songs_cleaned.csv")
print(f"Columns: {list(songs_preprocessed_df.columns)}")

Saved 697,392 rows → ../data/processed/songs_cleaned.csv
Columns: ['track_id', 'track_name', 'artist_name', 'genre', 'popularity', 'danceability', 'energy', 'loudness', 'speechiness', 'acousticness', 'instrumentalness', 'liveness', 'valence', 'tempo']


In [29]:
# Train/test split (80/20) — saved via pandas

songs_train_df, songs_test_df = train_test_split(songs_preprocessed_df, test_size=0.2, random_state=42)

songs_train_df.to_csv(f"{SPLIT_DIR}/songs_train.csv", index=False)
songs_test_df.to_csv(f"{SPLIT_DIR}/songs_test.csv",  index=False)

print(f"Train: {len(songs_train_df):,} rows → {SPLIT_DIR}/songs_train.csv")
print(f"Test:  {len(songs_test_df):,} rows  → {SPLIT_DIR}/songs_test.csv")

Train: 557,913 rows → ../outputs/train_test_split/songs_train.csv
Test:  139,479 rows  → ../outputs/train_test_split/songs_test.csv


2.5 - Clean ALS Interaction Data (Playlist Dataset)


In [30]:


# on_bad_lines='skip' drops rows where song/playlist names contain unescaped commas
users_ds = pd.read_csv(
    f"{DATA_DIR}/spotify_playlists/spotify_dataset.csv",
    quotechar='"',
    skipinitialspace=True,
    on_bad_lines='skip'
)

# Normalize column names — strip spaces and quotes
users_ds.columns = [c.strip().strip('"') for c in users_ds.columns]

# Drop nulls and duplicates
users_ds = users_ds.dropna(subset=["user_id", "trackname", "artistname"])
users_ds = users_ds .drop_duplicates(subset=["user_id", "trackname", "artistname"])

# Encode user_id → integer index
users_ds["user_id_enc"] = pd.factorize(users_ds["user_id"])[0]

# Encode track (name + artist combined) → integer index
users_ds["track_key"]   = users_ds["trackname"] + "||" + users_ds["artistname"]
users_ds["song_id_enc"] = pd.factorize(users_ds["track_key"])[0]

# Implicit interaction = 1
users_ds["interaction"] = 1.0

# Final columns
users_ds_final = users_ds[[
    "user_id_enc", "song_id_enc", "interaction",
    "user_id", "trackname", "artistname", "playlistname"
]].rename(columns={
    "user_id_enc":  "user_id",
    "song_id_enc":  "song_id",
    "user_id":      "user_id_original",
    "trackname":    "track_name",
    "artistname":   "artist_name",
    "playlistname": "playlist_name"
})

print(f"Interactions: {len(users_ds_final):,}")
print(f"Unique users: {users_ds_final['user_id'].nunique():,}")
print(f"Unique songs: {users_ds_final['song_id'].nunique():,}")
users_ds_final.head()

Interactions: 11,374,143
Unique users: 15,914
Unique songs: 2,790,802


,user_id,song_id,interaction,user_id_original,track_name,artist_name,playlist_name
0,0,0,1.0,9cc0cfd4d7d7885102480dd99e7a90d6,(The Angels Wanna Wear My) Red Shoes,Elvis Costello,HARD ROCK 2010
1,0,1,1.0,9cc0cfd4d7d7885102480dd99e7a90d6,"(What's So Funny 'Bout) Peace, Love And Unders...",Elvis Costello & The Attractions,HARD ROCK 2010
2,0,2,1.0,9cc0cfd4d7d7885102480dd99e7a90d6,7 Years Too Late,Tiffany Page,HARD ROCK 2010
3,0,3,1.0,9cc0cfd4d7d7885102480dd99e7a90d6,Accidents Will Happen,Elvis Costello & The Attractions,HARD ROCK 2010
4,0,4,1.0,9cc0cfd4d7d7885102480dd99e7a90d6,Alison,Elvis Costello,HARD ROCK 2010


In [31]:
users_ds_final.to_csv(f"{PROCESSED_DIR}/users_interactions_cleaned.csv", index=False)

print(f"Saved {len(users_ds_final):,} rows → {PROCESSED_DIR}/users_interactions_cleaned.csv")
print(f"Columns: {list(users_ds_final.columns)}")

Saved 11,374,143 rows → ../data/processed/users_interactions_cleaned.csv
Columns: ['user_id', 'song_id', 'interaction', 'user_id_original', 'track_name', 'artist_name', 'playlist_name']


2.6 - User Train/Test Split


In [32]:
# ALS train/test split (80/20)
# Split on interactions — ALS trains on 80% of user-song pairs,
# then tries to predict the held-out 20%

users_train_ds, users_test_ds = train_test_split(users_ds_final, test_size=0.2, random_state=42)

users_train_ds.to_csv(f"{SPLIT_DIR}/users_train.csv", index=False)
users_test_ds.to_csv(f"{SPLIT_DIR}/users_test.csv",   index=False)

print(f"ALS Train: {len(users_train_ds):,} rows → {SPLIT_DIR}/users_train.csv")
print(f"ALS Test:  {len(users_test_ds):,} rows  → {SPLIT_DIR}/users_test.csv")
print(f"Unique users in train: {users_train_ds['user_id'].nunique():,}")
print(f"Unique songs in train: {users_train_ds['song_id'].nunique():,}")
print(f"Unique users in test:  {users_test_ds['user_id'].nunique():,}")
print(f"Unique songs in test:  {users_test_ds['song_id'].nunique():,}")

ALS Train: 9,099,314 rows → ../outputs/train_test_split/users_train.csv
ALS Test:  2,274,829 rows  → ../outputs/train_test_split/users_test.csv
Unique users in train: 15,850
Unique songs in train: 2,430,136
Unique users in test:  15,391
Unique songs in test:  970,581


In [33]:
spark.stop()
print("Done. Spark session closed.")

Done. Spark session closed.
